In [5]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

df = pd.read_csv("../outputs/anomalies.csv")

geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry)

#base map indonesia
m = folium.Map(location=[-2.5489, 118.0149], zoom_start=5)

# anomali point
for _, row in gdf.iterrows():
    if row["anomaly_score"] == -1:
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            color="red",
            fill=True,
            fill_color="red",
            popup=f"Risk: {row['risk_score']:.2f}"
        ).add_to(m)

m.save("../outputs//maps/final_risk_map.html")

print("Map saved.")


Map saved.


In [6]:
import pandas as pd
import joblib
import os

from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../outputs/risk_scored.csv")
df.head()

features = df[
    [
        "latitude",
        "longitude",
        "magnitude",
        "depth",
        "risk_score"
    ]
]

dbscan_model = DBSCAN(
    eps=0.5,
    min_samples=10
)

dbscan_model.fit(
    features[
        ["latitude", "longitude"]
    ]
)

df["cluster"] = dbscan_model.labels_

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42
)

iso_model.fit(features)

df["anomaly_score"] = iso_model.predict(features)

os.makedirs("../models", exist_ok=True)

joblib.dump(
    dbscan_model,
    "../models/dbscan_model.pkl"
)

joblib.dump(
    iso_model,
    "../models/isolation_forest.pkl"
)

['../models/isolation_forest.pkl']